In [1]:
import numpy as np
import torch
import scipy.sparse as sp

print("Testing individual components...")

# Test 1: Basic numpy
print("\n[TEST 1] NumPy operations...")
try:
    arr = np.random.randn(100, 50)
    print(f"✓ NumPy works: {arr.shape}")
except:
    print("✗ NumPy FAILED")

# Test 2: Basic torch
print("\n[TEST 2] PyTorch operations...")
try:
    tensor = torch.randn(100, 50)
    print(f"✓ PyTorch works: {tensor.shape}")
except:
    print("✗ PyTorch FAILED")

# Test 3: Scipy sparse
print("\n[TEST 3] SciPy sparse operations...")
try:
    sparse_mat = sp.random(100, 100, density=0.1)
    print(f"✓ SciPy sparse works: {sparse_mat.shape}")
except:
    print("✗ SciPy sparse FAILED")

# Test 4: Torch sparse
print("\n[TEST 4] PyTorch sparse operations...")
try:
    indices = torch.LongTensor([[0, 1, 2], [1, 2, 0]])
    values = torch.FloatTensor([1, 2, 3])
    sparse_tensor = torch.sparse_coo_tensor(indices, values, (3, 3))
    print(f"✓ PyTorch sparse works: {sparse_tensor.shape}")
except:
    print("✗ PyTorch sparse FAILED")

# Test 5: to_dense on large sparse tensor
print("\n[TEST 5] Large sparse to dense conversion...")
try:
    size = 3605
    density = 0.01
    sparse_mat = sp.random(size, size, density=density, format='coo')
    indices = np.vstack((sparse_mat.row, sparse_mat.col))
    values = sparse_mat.data

    sparse_tensor = torch.sparse_coo_tensor(
        torch.LongTensor(indices),
        torch.FloatTensor(values),
        (size, size)
    )
    print(f"  Sparse tensor created: {sparse_tensor.shape}")

    # This might be the problem - converting large sparse to dense
    print(f"  Converting to dense (this uses {size*size*4/1e9:.2f} GB memory)...")
    dense = sparse_tensor.to_dense()
    print(f"✓ Large sparse to dense works: {dense.shape}")
except Exception as e:
    print(f"✗ Large sparse to dense FAILED: {e}")

# Test 6: The actual problematic line
print("\n[TEST 6] The actual data loading pattern...")
try:
    # Simulate your data loading
    adj = sp.random(3605, 3605, density=0.01, format='csr')
    adj_label = adj + sp.eye(adj.shape[0])

    # Convert to tuple
    if not sp.isspmatrix_coo(adj_label):
        adj_label = adj_label.tocoo()
    coords = np.vstack((adj_label.row, adj_label.col)).transpose()
    values = adj_label.data
    shape = adj_label.shape
    adj_label_tuple = (coords, values, shape)

    # Convert to tensor
    adj_label_tensor = torch.sparse_coo_tensor(
        indices=torch.LongTensor(adj_label_tuple[0].T),
        values=torch.FloatTensor(adj_label_tuple[1]),
        size=torch.Size(adj_label_tuple[2])
    ).coalesce()

    print(f"  Sparse tensor: {adj_label_tensor.shape}")

    # THIS IS LIKELY THE PROBLEM
    print(f"  Converting to dense view...")
    dense_view = adj_label_tensor.to_dense().view(-1)
    print(f"✓ Pattern works: {dense_view.shape}")

except Exception as e:
    print(f"✗ Pattern FAILED: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)
print("If all tests passed, the issue is in the model itself.")
print("If TEST 5 or TEST 6 failed, it's a memory/sparse issue.")
print("="*60)

Testing individual components...

[TEST 1] NumPy operations...
✓ NumPy works: (100, 50)

[TEST 2] PyTorch operations...
✓ PyTorch works: torch.Size([100, 50])

[TEST 3] SciPy sparse operations...
✓ SciPy sparse works: (100, 100)

[TEST 4] PyTorch sparse operations...
✓ PyTorch sparse works: torch.Size([3, 3])

[TEST 5] Large sparse to dense conversion...
  Sparse tensor created: torch.Size([3605, 3605])
  Converting to dense (this uses 0.05 GB memory)...
✓ Large sparse to dense works: torch.Size([3605, 3605])

[TEST 6] The actual data loading pattern...
  Sparse tensor: torch.Size([3605, 3605])
  Converting to dense view...
✓ Pattern works: torch.Size([12996025])

If all tests passed, the issue is in the model itself.
If TEST 5 or TEST 6 failed, it's a memory/sparse issue.


In [2]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import scipy.sparse as sp
from preprocessing import load_data, sparse_to_tuple, preprocess_graph, get_device
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt

print("="*60)
print("DIAGNOSTIC: Finding Why ARI Decreases")
print("="*60)

# Load data
dataset = "baron3"
nClusters = 14
adj, features, labels = load_data('baron3', './data/baron3', True)

print(f"\n[1] Data Statistics:")
print(f"  Nodes: {features.shape[0]}")
print(f"  Features: {features.shape[1]}")
print(f"  True clusters: {len(np.unique(labels))}")
print(f"  Label distribution: {np.bincount(labels)}")

# Check label distribution
label_counts = np.bincount(labels)
print(f"\n[2] Imbalance Analysis:")
print(f"  Smallest cluster: {label_counts.min()} samples")
print(f"  Largest cluster: {label_counts.max()} samples")
print(f"  Imbalance ratio: {label_counts.max() / label_counts.min():.1f}:1")
print(f"  Clusters with <10 samples: {np.sum(label_counts < 10)}")

# Test simple K-means on raw features
print(f"\n[3] Testing K-means on Raw Features:")
features_dense = features.toarray()

for n_clusters in [14, 10, 8, 5]:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    pred = kmeans.fit_predict(features_dense)
    ari = adjusted_rand_score(labels, pred)
    nmi = normalized_mutual_info_score(labels, pred)
    print(f"  K={n_clusters}: ARI={ari:.4f}, NMI={nmi:.4f}")

# Test with PCA reduced features
print(f"\n[4] Testing K-means on PCA Features:")
from sklearn.decomposition import PCA

for n_components in [50, 100, 200]:
    pca = PCA(n_components=n_components, random_state=42)
    features_pca = pca.fit_transform(features_dense)

    kmeans = KMeans(n_clusters=14, random_state=42, n_init=20)
    pred = kmeans.fit_predict(features_pca)
    ari = adjusted_rand_score(labels, pred)
    nmi = normalized_mutual_info_score(labels, pred)
    print(f"  PCA-{n_components}: ARI={ari:.4f}, NMI={nmi:.4f}")

# Test graph-based embedding
print(f"\n[5] Testing Simple Graph Embedding:")
from sklearn.manifold import SpectralEmbedding

try:
    # Use adjacency for spectral embedding
    adj_matrix = adj.toarray()

    for n_components in [32, 64]:
        spectral = SpectralEmbedding(n_components=n_components, random_state=42, affinity='precomputed')
        emb = spectral.fit_transform(adj_matrix)

        kmeans = KMeans(n_clusters=14, random_state=42, n_init=20)
        pred = kmeans.fit_predict(emb)
        ari = adjusted_rand_score(labels, pred)
        nmi = normalized_mutual_info_score(labels, pred)
        print(f"  Spectral-{n_components}: ARI={ari:.4f}, NMI={nmi:.4f}")
except Exception as e:
    print(f"  Spectral embedding failed: {e}")

# Analyze what happens with random embeddings
print(f"\n[6] Testing Random Embeddings (Baseline):")
for seed in [42, 123, 456]:
    np.random.seed(seed)
    random_emb = np.random.randn(features.shape[0], 32)

    kmeans = KMeans(n_clusters=14, random_state=42, n_init=20)
    pred = kmeans.fit_predict(random_emb)
    ari = adjusted_rand_score(labels, pred)
    nmi = normalized_mutual_info_score(labels, pred)
    print(f"  Random seed {seed}: ARI={ari:.4f}, NMI={nmi:.4f}")

# Check if labels are correct
print(f"\n[7] Label Sanity Check:")
print(f"  Label range: [{labels.min()}, {labels.max()}]")
print(f"  Expected range: [0, {nClusters-1}]")
print(f"  Labels dtype: {labels.dtype}")
print(f"  Number of unique labels: {len(np.unique(labels))}")

# Check adjacency matrix
print(f"\n[8] Adjacency Matrix Analysis:")
print(f"  Shape: {adj.shape}")
print(f"  Non-zeros: {adj.nnz}")
print(f"  Density: {adj.nnz / (adj.shape[0] * adj.shape[1]):.6f}")
print(f"  Is symmetric: {np.allclose(adj.toarray(), adj.toarray().T)}")

# Check for disconnected components
from scipy.sparse.csgraph import connected_components
n_components, component_labels = connected_components(adj, directed=False)
print(f"  Connected components: {n_components}")
if n_components > 1:
    print(f"  WARNING: Graph has {n_components} disconnected components!")
    component_sizes = np.bincount(component_labels)
    print(f"  Component sizes: {component_sizes}")

# Analyze feature distribution
print(f"\n[9] Feature Analysis:")
print(f"  Feature mean: {features_dense.mean():.4f}")
print(f"  Feature std: {features_dense.std():.4f}")
print(f"  Feature min: {features_dense.min():.4f}")
print(f"  Feature max: {features_dense.max():.4f}")
print(f"  % zeros: {100 * (features_dense == 0).sum() / features_dense.size:.2f}%")

# Final diagnosis
print(f"\n" + "="*60)
print("DIAGNOSIS:")
print("="*60)

# Get best k-means ARI
kmeans_best = KMeans(n_clusters=14, random_state=42, n_init=50)
pred_best = kmeans_best.fit_predict(features_dense)
ari_best = adjusted_rand_score(labels, pred_best)

print(f"\nBest K-means ARI on raw features: {ari_best:.4f}")

if ari_best < 0.3:
    print("\n⚠️ CRITICAL: K-means gets ARI < 0.3 on raw features")
    print("   This means:")
    print("   1. Your labels may not correspond to feature-based clusters")
    print("   2. Graph structure is essential (features alone don't work)")
    print("   3. The clustering task is extremely difficult")
    print("\n   Recommendations:")
    print("   - Check if labels are correct")
    print("   - Verify this is the right dataset")
    print("   - Consider that ARI 0.18 might be close to the maximum achievable")
elif ari_best < 0.5:
    print(f"\n✓ K-means gets moderate ARI ({ari_best:.4f})")
    print("   The problem is learnable but challenging")
    print("   Your neural network should be able to exceed this")
else:
    print(f"\n✓ K-means gets good ARI ({ari_best:.4f})")
    print("   The problem is learnable - neural network should work")

print("="*60)

DIAGNOSTIC: Finding Why ARI Decreases

[1] Data Statistics:
  Nodes: 3605
  Features: 1200
  True clusters: 14
  Label distribution: [1130  100   14    7   92   36    1  787   54  376    2    2  843  161]

[2] Imbalance Analysis:
  Smallest cluster: 1 samples
  Largest cluster: 1130 samples
  Imbalance ratio: 1130.0:1
  Clusters with <10 samples: 4

[3] Testing K-means on Raw Features:
  K=14: ARI=0.2393, NMI=0.4767
  K=10: ARI=0.2059, NMI=0.4495
  K=8: ARI=0.2125, NMI=0.4543
  K=5: ARI=0.1659, NMI=0.3816

[4] Testing K-means on PCA Features:
  PCA-50: ARI=0.2439, NMI=0.4813
  PCA-100: ARI=0.2407, NMI=0.4793
  PCA-200: ARI=0.2435, NMI=0.4801

[5] Testing Simple Graph Embedding:
  Spectral-32: ARI=0.5430, NMI=0.6909
  Spectral-64: ARI=0.1069, NMI=0.3967

[6] Testing Random Embeddings (Baseline):
  Random seed 42: ARI=-0.0000, NMI=0.0107
  Random seed 123: ARI=0.0003, NMI=0.0123
  Random seed 456: ARI=-0.0001, NMI=0.0102

[7] Label Sanity Check:
  Label range: [0, 13]
  Expected range: [

In [ ]:
for epochs_cluster in tqdm(epochs_cluster_0):
    for lr_cluster in lr_cluster_0:
        for alpha_recons in alpha_recons_0:
            for beta_cluster in beta_cluster_0:
                for gamma_structure in gamma_structure_0:
                    for embedding_size in embedding_size_0:
                        for delta_graph in delta_graph_0:
                           for num_neurons in num_neurons_0:
                              print(
                                  f"\n Starting training... with "
                                  f"epoch_cluster({epochs_cluster}) | "
                                  f"lr_cluster({lr_cluster}) | "
                                  f"alpha_recons({alpha_recons}) | "
                                  f"beta_cluster({beta_cluster}) | "
                                  f"gamma_gmcm({gamma_structure}) | "
                                  f"embedding_size({embedding_size}) | "
                                  f"delta_graph({delta_graph}) | "
                                  f"num_neurons({num_neurons})"
                              )
                              network = GMCM_VGAE_Final(
                                    adj=adj_norm,
                                    num_neurons=num_neurons,
                                    num_features=num_features,
                                    embedding_size=embedding_size,
                                    nClusters=nClusters,
                                    activation="Sigmoid",
                                    seed=seed,
                                    # Optimized loss weights
                                    alpha_recons=alpha_recons,     # Reconstruction
                                    beta_gmcm=beta_cluster,        # GMCM clustering
                                    gamma_zinb=gamma_structure,       # ZINB
                                    delta_graph=delta_graph      # Graph structure preservation
                                )
                              network.to(device)

                              start_time = time.perf_counter()

                              res, y_pred, y = network.train(
                                    acc_list=[],
                                    adj_norm=adj_norm.to(device),
                                    features=features.to(device),
                                    adj_label=adj_label.to(device),
                                    y=labels,
                                    weight_tensor=weight_tensor_orig.to(device),
                                    norm=norm,
                                    optimizer="Adam",
                                    epochs=epochs_cluster,
                                    lr=lr_cluster,
                                    save_path=save_path,
                                    dataset=dataset,
                                    features_new=features_new
                                )

                              result_dict["epoch_cluster"].append(epochs_cluster)
                              result_dict["lr_cluster"].append(lr_cluster)
                              result_dict["alpha_recons_cluster"].append(alpha_recons)
                              result_dict["beta_cluster"].append(beta_cluster)
                              result_dict["gamma_structure_cluster"].append(gamma_structure)
                              result_dict["delta_graph_cluster"].append(delta_graph)
                              result_dict["embedding_size_cluster"].append(embedding_size)
                              result_dict["num_neurons_cluster"].append(num_neurons)
                              result_dict["ACC"].append(res[0])
                              result_dict["ARI"].append(res[1])
                              result_dict["NMI"].append(res[2])
result_data=pd.DataFrame(result_dict)
result_data.to_csv(f"{save_path}{dataset}/cluster/results.csv")
